# Lab 1: Triage the Queue

**AI-2: AI Backend Engineering** | Northbrook Partners Support Pipeline

---

It's Monday morning. Northbrook Partners' internal support system went down over the weekend — and 20 unprocessed tickets are sitting in the queue. Help desk is slammed. Your manager pings you:

> *"Can you build something to auto-triage these? I need category, urgency, a clean summary, and routing for every ticket — structured, not vibes. And for the easy ones? Draft a response so we can clear them fast."*

**What you will build:**
1. A Pydantic schema for structured triage output
2. A triage function that classifies and extracts from a single ticket
3. A batch processor that handles the full queue with cost tracking
4. An analysis dashboard of your results
5. An auto-response pipeline for simple tickets using the IT knowledge base
6. A complete JSONL export of all triage results

**Rubric:**

| Section | Points |
|---------|--------|
| Schema Design | 15 |
| Triage Function | 25 |
| Batch Processing | 20 |
| Auto-Response & Export | 25 |
| Process Checks & Reflection | 15 |
| **Total** | **100** |

**Passing: 70+ points.** Partial credit for partially correct functions.

**Estimated time: 2–3 hours**

In [ ]:
import sys, json
from pathlib import Path
from pydantic import BaseModel, Field
from typing import Literal
from dotenv import load_dotenv
import anthropic

_here = Path(".").resolve()
_root = _here
while _root != _root.parent:
    if (_root / "pyproject.toml").exists():
        break
    _root = _root.parent
sys.path.insert(0, str(_root))
load_dotenv()

client = anthropic.Anthropic()

print("Setup complete.")

In [ ]:
# Load the ticket queue
TICKETS_PATH = _root / "data" / "northbrook" / "support_tickets.json"
tickets = json.loads(TICKETS_PATH.read_text())
print(f"Loaded {len(tickets)} tickets\n")

# Preview one ticket
print(f"--- {tickets[0]['ticket_id']} ---")
print(f"From: {tickets[0]['submitted_by']} ({tickets[0]['department']})")
print(f"Subject: {tickets[0]['subject']}")
print(f"Body: {tickets[0]['body'][:200]}...")

In [ ]:
# Skim the queue — what are we dealing with?
for t in tickets:
    print(f"  {t['ticket_id']:>6}  {t['department']:<12}  {t['subject'][:60]}")

## Section 1: Design Your Schema

**Skills tested:** Pydantic models (Session 1.2)

Design a Pydantic model that captures the structured triage output for a single ticket. Think about what fields a help desk manager would need to route and prioritize work.

**Required fields** (you choose the types and constraints):

| Field | Purpose |
|-------|--------|
| `ticket_id` | Which ticket this is |
| `category` | What kind of issue (IT, Security, Facilities, HR, Finance, Other) |
| `urgency` | How urgent (critical, high, medium, low) |
| `summary` | 1-2 sentence clean summary of the issue |
| `routing_team` | Which team should handle this |
| `key_details` | Important specifics extracted from the ticket |
| `auto_resolvable` | Can this ticket be resolved with a knowledge base article and a drafted response, or does it require human investigation? |

You may add additional fields if you think they'd be useful. Use `Field(description=...)` — these descriptions help Claude understand what you want.

In [ ]:
class TriageResult(BaseModel):
    """Structured triage output for a single support ticket."""

    # -- Define your fields here ------------------------------------------
    # Use Field(description="...") to guide Claude's extraction.
    # Use Literal["option1", "option2"] to constrain categorical fields.
    #
    # Required fields:
    #   ticket_id: str
    #   category: constrain to "IT", "Security", "Facilities", "HR", "Finance", "Other"
    #   urgency: constrain to "critical", "high", "medium", "low"
    #   summary: str (1-2 sentences)
    #   routing_team: str
    #   key_details: list of strings
    #   auto_resolvable: bool — can this be answered from a standard KB article?
    #
    # Optional ideas: sentiment, requires_response (bool), estimated_effort

    # YOUR CODE HERE
    pass

In [ ]:
# Quick schema check — does it serialize correctly?
print(json.dumps(TriageResult.model_json_schema(), indent=2))

### Process Check

**R1:** What fields did you consider but leave out? What helped you decide what belongs in the schema vs. what stays in the prompt? (2-3 sentences)

*Your answer:*

## Section 2: Build the Triage Function

**Skills tested:** API calls (Session 1.1), Structured output (Session 1.2)

Write a function that takes a raw ticket dict and returns a `TriageResult`. Use `client.messages.parse()` with `output_format` — the same pattern from Session 1.2.

**Reminder — the SDK structured output pattern:**
```python
response = client.messages.parse(
    model="claude-sonnet-4-5",
    max_tokens=1024,
    temperature=0.0,
    system="your system prompt",
    messages=[{"role": "user", "content": "your message"}],
    output_format=YourPydanticClass,
)
result = response.parsed_output  # typed Pydantic instance
tokens_in = response.usage.input_tokens
tokens_out = response.usage.output_tokens
```

In [ ]:
TRIAGE_SYSTEM_PROMPT = (
    "You are an IT support triage specialist for Northbrook Partners. "
    "Classify and extract structured data from support tickets accurately. "
    "Be concise in summaries. Route to the most appropriate team."
    # -- Add guidance for auto_resolvable -----------------------------------
    # Tell Claude WHEN a ticket should be marked auto_resolvable.
    # Think: what kinds of tickets can be answered with a standard KB article
    # (password resets, known troubleshooting steps, policy questions)?
    # What kinds require human investigation (outages, data loss, security
    # incidents, vague requests with missing info)?
)


def triage_ticket(ticket: dict) -> dict:
    """Classify and extract structured data from a single support ticket.

    Args:
        ticket: raw ticket dict with keys: ticket_id, submitted_by,
                department, submitted_at, subject, body

    Returns:
        dict with keys: "result" (TriageResult), "input_tokens" (int),
        "output_tokens" (int)
    """
    # -- Step 1 -------------------------------------------------------
    # Build the user message from the ticket fields.
    # Include: ticket_id, submitted_by, department, subject, body
    # Format it clearly so Claude can parse it.

    # YOUR CODE HERE


    # -- Step 2 -------------------------------------------------------
    # Call client.messages.parse() with:
    #   model="claude-sonnet-4-5"
    #   max_tokens=1024
    #   temperature=0.0
    #   system=TRIAGE_SYSTEM_PROMPT
    #   messages=[{"role": "user", "content": your_message}]
    #   output_format=TriageResult

    # YOUR CODE HERE


    # -- Step 3 -------------------------------------------------------
    # Return a dict with the parsed result and token usage.

    return {"result": ..., "input_tokens": ..., "output_tokens": ...}

In [ ]:
# Test on the first ticket
test = triage_ticket(tickets[0])
r = test["result"]

print(f"Ticket:   {r.ticket_id}")
print(f"Category: {r.category}")
print(f"Urgency:  {r.urgency}")
print(f"Summary:  {r.summary}")
print(f"Route to: {r.routing_team}")
print(f"Details:  {r.key_details}")
print(f"\nTokens: {test['input_tokens']} in / {test['output_tokens']} out")

### Process Check

**R2:** Look at the result above. Try running `triage_ticket()` on 2-3 more tickets (e.g., `tickets[5]`, `tickets[15]`). Did anything get classified in a way you didn't expect? What would you adjust in your system prompt? (2-3 sentences)

*Your answer:*

## Section 3: Batch Processing

**Skills tested:** Batch processing patterns (Session 1.2)

Process all 20 tickets. Track successes, failures, and total cost. A real pipeline doesn't crash on one bad ticket — wrap each call so failures are captured, not fatal.

In [ ]:
def triage_batch(tickets: list[dict]) -> dict:
    """Process all tickets and return structured results with stats.

    Args:
        tickets: list of raw ticket dicts

    Returns:
        dict with keys:
            "results": list of TriageResult objects (successful only)
            "errors": list of dicts with "ticket_id" and "error"
            "total_input_tokens": int
            "total_output_tokens": int
    """
    results = []
    errors = []
    total_input = 0
    total_output = 0

    # -- Step 1 -------------------------------------------------------
    # Loop through each ticket.
    # Call triage_ticket() for each one.
    # Wrap in try/except to catch failures without crashing.
    # On success: append the TriageResult to results, add to token totals.
    # On failure: append {"ticket_id": ..., "error": str(e)} to errors.
    # Print progress as you go (e.g., "Processing NB-001... done")

    # YOUR CODE HERE

    return {
        "results": results,
        "errors": errors,
        "total_input_tokens": total_input,
        "total_output_tokens": total_output,
    }

In [ ]:
batch = triage_batch(tickets)
print(f"\nProcessed: {len(batch['results'])} succeeded, {len(batch['errors'])} failed")
print(f"Total tokens: {batch['total_input_tokens']:,} in / {batch['total_output_tokens']:,} out")
if batch["errors"]:
    print(f"\nErrors:")
    for e in batch["errors"]:
        print(f"  {e['ticket_id']}: {e['error']}")

### Process Check

**R3:** Based on your cost numbers above, what would it cost to triage 10,000 tickets? Would you change anything about the pipeline at that scale? (2-3 sentences)

*Your answer:*

## Section 4: Analysis

**Skills tested:** Pipeline evaluation, cost awareness

Build a summary dashboard from your batch results. Then answer two analysis questions.

In [ ]:
def build_report(results: list, total_input: int, total_output: int) -> None:
    """Print a formatted triage report from batch results.

    Display:
    1. Category breakdown (count per category)
    2. Urgency breakdown (count per urgency level)
    3. Routing summary (count per routing team)
    4. Cost estimate (using $3.00/M input, $15.00/M output for Sonnet 4.5)
    """
    # -- Step 1 -------------------------------------------------------
    # Count tickets per category, per urgency, and per routing team.
    # Hint: use a dict to accumulate counts.

    # YOUR CODE HERE


    # -- Step 2 -------------------------------------------------------
    # Print the breakdowns in a readable format.

    # YOUR CODE HERE


    # -- Step 3 -------------------------------------------------------
    # Calculate and print cost estimate.
    # Sonnet 4.5 pricing: $3.00 per million input tokens,
    #                      $15.00 per million output tokens.

    # YOUR CODE HERE
    pass

In [ ]:
build_report(batch["results"], batch["total_input_tokens"], batch["total_output_tokens"])

## Section 5: Auto-Response Pipeline

**Skills tested:** Prompt design, context injection, cost awareness

Some tickets are straightforward — password resets, known troubleshooting steps, policy questions. For these, you can draft an automated response using Northbrook's internal IT knowledge base.

Your job:
1. Load the knowledge base articles
2. Filter your batch results to only the `auto_resolvable` tickets
3. For each one, send the ticket **plus the entire knowledge base** to Claude and draft a response

**Important:** You are sending ALL knowledge base articles as context for every ticket. This is intentional — we'll revisit why this matters in the final reflection.

In [ ]:
# Load the knowledge base
KB_DIR = _root / "data" / "northbrook" / "knowledge_base"

kb_docs = {}
for f in sorted(KB_DIR.glob("*.md")):
    kb_docs[f.name] = f.read_text()

print(f"Loaded {len(kb_docs)} knowledge base articles:\n")
for name, text in kb_docs.items():
    print(f"  {name} ({len(text):,} chars)")

# Build the full KB context string — ALL articles, every time
kb_context = "\n\n---\n\n".join(
    f"## {name}\n\n{text}" for name, text in kb_docs.items()
)
print(f"\nTotal KB context: {len(kb_context):,} characters")

In [ ]:
# Filter to auto-resolvable tickets
auto_tickets = [r for r in batch["results"] if r.auto_resolvable]
ticket_lookup = {t["ticket_id"]: t for t in tickets}

print(f"{len(auto_tickets)} of {len(batch['results'])} tickets marked auto-resolvable:\n")
for r in auto_tickets:
    print(f"  {r.ticket_id}  [{r.category}]  {r.summary[:70]}")

In [ ]:
def draft_response(triage: TriageResult, original_ticket: dict,
                   kb_context: str) -> dict:
    """Draft a response for an auto-resolvable ticket using the knowledge base.

    Args:
        triage: the TriageResult from the triage step
        original_ticket: the raw ticket dict
        kb_context: the full knowledge base as a single string

    Returns:
        dict with keys: "ticket_id", "response", "input_tokens", "output_tokens"
    """
    # -- Step 1 -------------------------------------------------------
    # Design a system prompt for response drafting.
    # It should instruct Claude to:
    #   - Write a helpful, professional response to the employee
    #   - Use ONLY information from the provided knowledge base
    #   - Reference specific steps or policies from the KB
    #   - Keep the tone friendly and concise

    # YOUR CODE HERE — define RESPONSE_SYSTEM_PROMPT as a string


    # -- Step 2 -------------------------------------------------------
    # Build the user message. Include:
    #   - The original ticket (submitter, subject, body)
    #   - The triage summary and category
    #   - The full KB context
    # Hint: f"Ticket from {original_ticket['submitted_by']}:\n..."
    #        f"\n\nKnowledge Base:\n\n{kb_context}"

    # YOUR CODE HERE


    # -- Step 3 -------------------------------------------------------
    # Call client.messages.create() (not .parse — we want free text).
    #   model="claude-haiku-4-5"  (cheaper for response drafting)
    #   max_tokens=1024
    #   temperature=0.3
    #   system=RESPONSE_SYSTEM_PROMPT
    #   messages=[{"role": "user", "content": your_message}]

    # YOUR CODE HERE


    # -- Step 4 -------------------------------------------------------
    # Return the result dict.

    return {"ticket_id": ..., "response": ..., "input_tokens": ..., "output_tokens": ...}

In [ ]:
# Run auto-response for all auto-resolvable tickets
responses = []
for r in auto_tickets:
    original = ticket_lookup[r.ticket_id]
    print(f"  Drafting response for {r.ticket_id}...", end=" ")
    resp = draft_response(r, original, kb_context)
    responses.append(resp)
    print(f"done ({resp['input_tokens']:,} in / {resp['output_tokens']:,} out)")

total_in = sum(r["input_tokens"] for r in responses)
total_out = sum(r["output_tokens"] for r in responses)
print(f"\nDrafted {len(responses)} responses")
print(f"Auto-response tokens: {total_in:,} in / {total_out:,} out")

In [ ]:
# Review the drafted responses
for resp in responses:
    original = ticket_lookup[resp["ticket_id"]]
    print(f"\n{'=' * 60}")
    print(f"TICKET {resp['ticket_id']}: {original['subject']}")
    print(f"FROM: {original['submitted_by']} ({original['department']})")
    print(f"{'-' * 60}")
    print(f"DRAFTED RESPONSE:\n\n{resp['response']}")
    print(f"\n[Tokens: {resp['input_tokens']:,} in / {resp['output_tokens']:,} out]")
    print(f"{'=' * 60}")

### Process Check

**R4:** Pick the weakest auto-response above. What made it weak — the system prompt, the knowledge base content, or the ticket itself? What would you change? (2-3 sentences)

*Your answer:*

## Section 6: Export All Results

Save all 20 triage results to a single JSONL file. For tickets that received a draft response, include it in the record.

In [ ]:
# Export ALL triage results to a single JSONL file
# Auto-resolvable tickets include their draft response
output_path = _here / "lab_1_triage_output.jsonl"
response_lookup = {r["ticket_id"]: r["response"] for r in responses}

with open(output_path, "w") as f:
    for r in batch["results"]:
        record = r.model_dump()
        if r.ticket_id in response_lookup:
            record["draft_response"] = response_lookup[r.ticket_id]
        f.write(json.dumps(record) + "\n")

print(f"Exported {len(batch['results'])} triage results to {output_path.name}")
print(f"  {len(response_lookup)} include draft responses\n")

# Preview
with open(output_path) as f:
    for i, line in enumerate(f):
        rec = json.loads(line)
        has_resp = "draft_response" in rec
        status = "  + response" if has_resp else ""
        print(f"  {rec['ticket_id']}  {rec['category']:<10}  {rec['urgency']:<8}{status}")

## Section 7: Final Reflection

Answer each question in 3-4 sentences based on your full results.

**Q1:** You sent the ENTIRE knowledge base as context for every auto-response. Look at your token counts — how many input tokens per response? What percentage of those tokens do you think were actually relevant to the specific ticket being answered? (3-4 sentences)

*Your answer:*

**Q2:** Imagine you could automatically select only the 1-2 most relevant knowledge base articles for each ticket instead of sending all of them. How would that change your costs? How would it change the quality of the responses? How might you build a system that picks the right articles automatically? (3-4 sentences)

*Your answer:*